# ESG Sustainability Agent — GRPO Training with TRL + OpenEnv

**OpenEnv Hackathon Submission | ESG Compliance & Sustainability Environment**

This notebook trains an LLM to act as a **corporate sustainability strategist** using
**Group Relative Policy Optimization (GRPO)** with verifiable rewards from a custom
ESG simulation environment.

**Stack follows official OpenEnv reference pattern:**
- TRL `GRPOTrainer` with `rollout_func` (same as Wordle/BlackJack reference tutorials)
- Unsloth 4-bit LoRA for memory-efficient fine-tuning on T4
- Four verifiable reward signals (outcome, format, anti-cheat, task completion)
- Hugging Face Hub push with benchmark evidence

| Step | What happens | Time |
|------|-------------|------|
| 1 | GPU check | 10s |
| 2 | Install deps | 4 min |
| 3 | Login to HF | 30s |
| 4 | Clone & verify env | 30s |
| 5 | Smoke test pipeline | 30s |
| 6 | Define rollout + rewards | instant |
| 7 | Load model (Unsloth) | 2 min |
| 8 | GRPO training | ~45 min |
| 9 | Benchmark & plots | 3 min |
| 10 | Push model to Hub | 2 min |

## Cell 1 — GPU Check

In [ ]:
import subprocess, sys, torch

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'WARNING: No GPU detected!')

assert torch.cuda.is_available(), 'STOP: Go to Runtime > Change runtime type > T4 GPU'
print(f'GPU  : {torch.cuda.get_device_name(0)}')
print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print('Ready.')

## Cell 2 — Install Dependencies

> **Important:** Install Unsloth FIRST — it pins a compatible TRL version automatically.
> Do NOT force-reinstall TRL separately (breaks Unsloth's requirements).

In [ ]:
# 1. Core deps
!pip install -q pydantic openai pyyaml datasets huggingface_hub

# 2. Unsloth — pulls pinned TRL version automatically
!pip install -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'

# 3. Remaining training deps (do NOT pin trl here)
!pip install -q peft accelerate bitsandbytes

# 4. Mock optional TRL deps that use removed pydantic v1 APIs
import sys
import types as _types
for _pkg in ['llm_blender', 'vllm']:
    try:
        __import__(_pkg)
    except Exception:
        _mod = _types.ModuleType(_pkg)
        _mod.__spec__ = None
        sys.modules[_pkg] = _mod
        print(f'Mocked: {_pkg}')
    try:
        __import__(_pkg)
    except Exception:
        sys.modules[_pkg] = MagicMock()
        print(f'Mocked optional dep: {_pkg}')

# 5. Verify GRPOTrainer imports correctly
from trl import GRPOConfig, GRPOTrainer
import trl
print(f'TRL {trl.__version__} | GRPOTrainer: OK')
print('All dependencies ready.')

## Cell 3 — Login to Hugging Face

In [ ]:
from huggingface_hub import notebook_login, HfApi

# This will show a secure token input widget
notebook_login()

# Verify login
api = HfApi()
user = api.whoami()
HF_USERNAME = user['name']
print(f'Logged in as: {HF_USERNAME}')
print(f'Model will be pushed to: {HF_USERNAME}/esg-rl-agent-grpo')

## Cell 4 — Clone Repository & Verify Environment

In [ ]:
import os

REPO_DIR = '/content/open_ENV'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/TharunBabu-05/OPEN-ENV.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
    print('Repo already present - pulled latest.')

%cd {REPO_DIR}

# Quick sanity check
!python -c "from env import ESGEnvironment; from tasks import TASKS; print('Tasks:', list(TASKS.keys()))"
print('Environment verified.')

## Cell 5 — Smoke Test (Validates Reward Pipeline)

In [ ]:
# Validates reward functions + dataset builder without loading a model
!python train_rl.py --smoke_test
print('If SMOKE TEST PASSED above, proceed to Cell 6.')

## Cell 6 — Define Rollout Function & Reward Signals

This follows the **official OpenEnv TRL pattern** (same as Wordle/BlackJack tutorials):
- `rollout_func` runs full ESG episodes and collects named reward signals
- Multiple independent reward functions (verifiable, no learned reward model)
- GRPOTrainer receives `rollout_func` directly (not a dataset of prompts)

In [ ]:
import json, re
from env import ESGEnvironment
from tasks import TASKS
from models import Action

# ── System prompt ─────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """You are an expert corporate sustainability strategist.
Each month you manage ESG (Environmental, Social, Governance) decisions for a company.

AVAILABLE ACTIONS (pick one number):
0=Solar panels, 1=Wind turbines, 2=Employee training, 3=Community programs,
4=Board diversity, 5=Emissions audit, 6=Green supply chain, 7=Water recycling,
8=No action (save budget)

RESPONSE FORMAT (strict JSON only):
{"action": <0-8>, "reasoning": "<brief justification>"}

Prioritise actions that improve ESG scores while respecting the budget."""

# ── Rollout function (official OpenEnv pattern) ───────────────────────────────
def rollout_func(prompts, trainer=None):
    """
    Runs full ESG episodes. Called automatically by GRPOTrainer each step.
    Returns named reward columns that map to individual reward_funcs below.
    """
    outcome_rewards, format_rewards, anticheat_rewards, completion_rewards = [], [], [], []

    for _ in prompts:
        task = TASKS['basic_compliance']
        env = ESGEnvironment(task, seed=42)
        obs = env.reset()
        ep_outcome, ep_format, ep_anticheat = 0.0, 0.0, 0.0
        prev_action = None

        for step in range(task.max_steps):
            # Build prompt from current observation
            user_msg = (
                f"Month {obs.month}/{task.max_steps} | "
                f"Budget: ${obs.budget_remaining:,.0f} | "
                f"ESG: E={obs.environmental_score:.1f} S={obs.social_score:.1f} "
                f"G={obs.governance_score:.1f}\n"
                f"Choose your ESG action (JSON only):"
            )
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_msg},
            ]

            # Generate using trainer's vllm/generation method
            if trainer is not None:
                from trl.experimental.openenv import generate_rollout_completions
                prompt_text = tokenizer.apply_chat_template(
                    messages, add_generation_prompt=True, tokenize=False
                )
                out = generate_rollout_completions(trainer, [prompt_text])[0]
                text = out.get('text') or tokenizer.decode(out['completion_ids'], skip_special_tokens=True)
            else:
                text = '{"action": 0, "reasoning": "default"}'

            # Parse action
            try:
                parsed = json.loads(re.search(r'\{.*?\}', text, re.DOTALL).group())
                action_id = int(parsed.get('action', 8))
                action_id = max(0, min(8, action_id))
                fmt_reward = 1.0
            except Exception:
                action_id = 8
                fmt_reward = 0.0

            # Anti-cheat: penalise repeated actions
            cheat_reward = -0.5 if action_id == prev_action else 0.0
            prev_action = action_id

            action = Action(action_type=action_id)
            result = env.step(action)
            obs = result.observation

            ep_outcome += float(result.reward.composite_reward if result.reward else 0.0)
            ep_format += fmt_reward
            ep_anticheat += cheat_reward

            if result.done:
                break

        steps_done = step + 1
        task_done = 1.0 if getattr(result, 'done', False) and ep_outcome > 0 else 0.0

        outcome_rewards.append(ep_outcome / steps_done)
        format_rewards.append(ep_format / steps_done)
        anticheat_rewards.append(ep_anticheat / steps_done)
        completion_rewards.append(task_done)

    return {
        "outcome_reward":    outcome_rewards,
        "format_reward":     format_rewards,
        "anticheat_reward":  anticheat_rewards,
        "completion_reward": completion_rewards,
    }

# ── Reward functions (one per signal — official pattern) ─────────────────────
def reward_outcome(completions, **kwargs):
    return [float(r) for r in kwargs.get("outcome_reward", [0.0]*len(completions))]

def reward_format(completions, **kwargs):
    return [float(r) for r in kwargs.get("format_reward", [0.0]*len(completions))]

def reward_anticheat(completions, **kwargs):
    return [float(r) for r in kwargs.get("anticheat_reward", [0.0]*len(completions))]

def reward_completion(completions, **kwargs):
    return [float(r) for r in kwargs.get("completion_reward", [0.0]*len(completions))]

print('Rollout function and reward signals defined.')
print('Reward signals: outcome | format | anticheat | completion')

## Cell 7 — Load Model with Unsloth (4-bit LoRA)

In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL_NAME = 'unsloth/Qwen2.5-0.5B-Instruct'  # Fast, fits T4 (15.6 GB)
MAX_SEQ_LEN = 512

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
model.print_trainable_parameters()

gpu_stats = torch.cuda.get_device_properties(0)
vram = round(gpu_stats.total_memory / 1e9, 1)
print(f'GPU: {gpu_stats.name} | VRAM: {vram} GB')

## Cell 8 — GRPO Training (~45 min on T4)

Uses `rollout_func` (official OpenEnv pattern) — the trainer runs full ESG episodes
and learns from 4 independent verifiable reward signals.

In [ ]:
import time, os
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

OUTPUT_DIR = 'outputs/esg_rl_trained'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Minimal dataset — prompts are placeholders; rollout_func drives the actual episodes
dataset = Dataset.from_dict({"prompt": ["Play ESG sustainably."] * 200})

grpo_config = GRPOConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    max_steps=150,
    learning_rate=5e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_new_tokens=128,
    max_prompt_length=400,
    temperature=0.7,
    warmup_steps=10,
    logging_steps=10,
    save_steps=50,
    report_to="none",
    remove_unused_columns=False,
    log_completions=True,
    seed=42,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[reward_outcome, reward_format, reward_anticheat, reward_completion],
    args=grpo_config,
    train_dataset=dataset,
    rollout_func=rollout_func,
)

print('Starting GRPO training (150 steps, ~45 min on T4)...')
start = time.time()
trainer_stats = trainer.train()
elapsed = time.time() - start

print(f'Training complete in {elapsed/60:.1f} min')
print(f'Loss: {trainer_stats.metrics.get("train_loss", "N/A")}')

# Save LoRA adapter
adapter_path = f'{OUTPUT_DIR}/lora_adapter'
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f'Adapter saved -> {adapter_path}')
print('Adapter files:', os.listdir(adapter_path))

## Cell 9 — Benchmark: Baseline vs Trained

In [ ]:
import os, json
from IPython.display import Image, display

# Heuristic baseline
!python benchmark.py --mode heuristic --seeds 42 43 44 --output results/baseline_heuristic.json

# Trained model (if adapter exists)
adapter = 'outputs/esg_rl_trained/lora_adapter'
if os.path.exists(adapter):
    !python benchmark.py --mode llm --model_path {adapter} --seeds 42 43 44 --output results/trained.json
    !python plot_results.py --baseline results/baseline_heuristic.json --trained results/trained.json --output_dir results
else:
    !python plot_results.py --baseline results/baseline_heuristic.json --output_dir results

# Print scores
baseline = json.load(open('results/baseline_heuristic.json'))
print(f'Heuristic baseline: {baseline["overall_mean_score"]:.3f}')
if os.path.exists('results/trained.json'):
    trained = json.load(open('results/trained.json'))
    delta = trained['overall_mean_score'] - baseline['overall_mean_score']
    print(f'Trained model:      {trained["overall_mean_score"]:.3f}')
    print(f'Delta:              {delta:+.3f}')

# Show plots
for plot in ['score_comparison.png', 'reward_history.png', 'esg_metrics.png']:
    if os.path.exists(f'results/{plot}'):
        display(Image(f'results/{plot}'))

## Cell 10 — Push Model + Evidence to HuggingFace Hub

In [ ]:
import os
from huggingface_hub import HfApi, create_repo

# HF_USERNAME was set in Cell 3 via notebook_login()
MODEL_REPO  = f'{HF_USERNAME}/esg-rl-agent-grpo'
ADAPTER_DIR = 'outputs/esg_rl_trained/lora_adapter'

assert os.path.exists(ADAPTER_DIR), f'Adapter not found at {ADAPTER_DIR} -- did Cell 8 complete?'

api = HfApi()

# Create repo
create_repo(MODEL_REPO, repo_type='model', exist_ok=True, private=False)
print(f'Repo: https://huggingface.co/{MODEL_REPO}')

# Push LoRA adapter
api.upload_folder(
    folder_path=ADAPTER_DIR,
    repo_id=MODEL_REPO,
    repo_type='model',
    commit_message='Add ESG RL GRPO trained LoRA adapter (OpenEnv Hackathon)',
)
print(f'[OK] Adapter pushed')

# Push benchmark evidence
for f in ['results/baseline_heuristic.json', 'results/trained.json',
          'results/score_comparison.png', 'results/reward_history.png', 'results/esg_metrics.png']:
    if os.path.exists(f):
        api.upload_file(path_or_fileobj=f, path_in_repo=f, repo_id=MODEL_REPO, repo_type='model')
        print(f'[OK] {f}')

print(f'\nDone! Model: https://huggingface.co/{MODEL_REPO}')